# Workshop 2: นับวัตถุตามสี (HSV)
ขั้นตอน: **โหลดภาพ → แปลงเป็น HSV → ตั้งช่วงสี → สร้าง Mask → Bitwise And → หา Contour → วาดผล**  แล้วกรองตามพื้นที่

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))   # ให้หาไฟล์ vision_camera.py เจอ

import cv2
import numpy as np
from vision_camera import show, grab_one, img_path, put_label
print("OpenCV", cv2.__version__, "พร้อมใช้งาน")

## 1. ฟังก์ชันนับวัตถุ (7 ขั้นตอน)

In [ ]:
def process(img, h_low, h_high, min_area, s_min=50, v_min=50):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)                        # 2. แปลงเป็น HSV
    lower = np.array([h_low, s_min, v_min])                           # 3. ช่วงสี
    upper = np.array([h_high, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)                             # 4. Mask ขาว-ดำ
    result = cv2.bitwise_and(img, img, mask=mask)                     # 5. เหลือเฉพาะสีที่เลือก
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_NONE)             # 6. หา Contour
    draw = img.copy()
    cv2.drawContours(draw, contours, -1, (255, 0, 0), 2)              # 7. วาดเส้นขอบ

    big = 0                                                           # กรองตามพื้นที่
    for c in contours:
        if cv2.contourArea(c) > min_area:
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(result, (x, y), (x + w, y + h), (200, 0, 100), 3)
            big += 1
    return draw, result, mask, len(contours), big

## 2. เลื่อนแถบเพื่อนับ
เริ่มที่สีแดง **H 0-10** แล้วปรับ **Min Area** ให้นับเฉพาะชิ้นใหญ่ได้ 5 ชิ้น
ลองนับสีอื่น: เหลือง ~20-35, เขียว ~50-80, น้ำเงิน ~95-120

In [ ]:
from ipywidgets import interact
import time

img = cv2.imread(img_path("shapes.png"))

def count(h_low=0, h_high=10, s_min=50, v_min=50, min_area=1000):
    t0 = time.perf_counter()
    draw, result, mask, n_all, n_big = process(img, h_low, h_high, min_area, s_min, v_min)
    ms = (time.perf_counter() - t0) * 1000
    print(f"วัตถุทั้งหมด: {n_all}   ใหญ่กว่า {min_area}: {n_big}   ใช้เวลา {ms:.1f} ms")
    show(draw, mask, result, titles=["เส้นขอบ", "Mask", "ผลลัพธ์"], size=5)

interact(count, h_low=(0, 179), h_high=(0, 179), s_min=(0, 255), v_min=(0, 255),
         min_area=(0, 5000, 100));

## 3. โจทย์ระดับ 2: สีแดงมี 2 ช่วง
สีแดงอยู่ทั้ง **H 0-10** และ **H 170-179** (วงล้อสีวนกลับมา) ลองรวม mask ทั้งสองช่วงด้วย `cv2.bitwise_or`

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
mask1 = cv2.inRange(hsv, np.array([0, 50, 50]), np.array([10, 255, 255]))
mask2 = cv2.inRange(hsv, np.array([170, 50, 50]), np.array([179, 255, 255]))
red = cv2.bitwise_or(mask1, mask2)
show(mask1, mask2, red, titles=["H 0-10", "H 170-179", "รวมกัน"], size=4)

## 4. Circularity: กลมแค่ไหน
สูตร **4π × พื้นที่ ÷ เส้นรอบรูป²**  วงกลมสมบูรณ์ = 1.0  สี่เหลี่ยม ≈ 0.79  สามเหลี่ยม ≈ 0.60

In [ ]:
_, _, mask, _, _ = process(img, 0, 10, 0)
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
out = img.copy()
circles = 0
for c in contours:
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    circ = 4 * np.pi * area / (peri ** 2)
    x, y, w, h = cv2.boundingRect(c)
    put_label(out, f"{circ:.2f}", org=(x, y - 5), scale=0.5)
    if circ > 0.85:
        circles += 1
print("วงกลมสีแดง:", circles, "วง")
show(out, titles=["ค่า Circularity ของวัตถุสีแดง"])

## 5. ถ่ายจากกล้อง Basler จำลอง

ที่ผ่านมาเราอ่านภาพจาก **ไฟล์** ส่วนนี้เปลี่ยนมาอ่านจาก **กล้อง** โดยใช้กล้องจำลองของ Basler ซึ่งทำงานเหมือนกล้องจริงทุกอย่าง แต่ส่งภาพจากไฟล์ `shapes.png` ออกมาแทนภาพจากเลนส์

**cell ด้านล่างทำ 4 อย่าง**

| บรรทัด | ทำอะไร |
|---|---|
| `SOURCE = "emu"` | เลือกแหล่งภาพ `"emu"` = กล้องจำลอง  `"basler"` = กล้องจริง  `"file"` = ไฟล์ |
| `frame = grab_one(...)` | **เปิดกล้อง → ตั้งค่า → ถ่าย 1 ภาพ → ปิดกล้อง** ได้ภาพเก็บไว้ในตัวแปร `frame` |
| `process(frame, 0, 10, 1000)` | นับวัตถุสีแดง (H 0–10) ที่ใหญ่กว่า 1000 พิกเซล ด้วยฟังก์ชันเดียวกับข้อ 1 |
| `show(...)` | แสดงภาพจากกล้องคู่กับผลการนับ |

ค่าที่ส่งให้ `grab_one`:
- `image="shapes.png"` ภาพที่ให้กล้องจำลองส่งออกมา (กล้องจริงไม่ใช้ค่านี้)
- `exposure_us=5000` เวลาเปิดรับแสง 5000 ไมโครวินาที (5 มิลลิวินาที)

**ทำไมต้องปิดกล้องทุกครั้ง:** กล้องเปิดได้ทีละโปรแกรม ถ้าเปิดค้างไว้ pylon Viewer หรือ notebook อื่นจะใช้กล้องไม่ได้ `grab_one` จึงปิดกล้องให้เสมอ แม้เกิด error

**สำคัญ:** ถ้าจะสลับระหว่าง `"emu"` กับ `"basler"` ให้กด **Restart** (ปุ่มลูกศรวนด้านบน) ก่อน เพราะโหมดจำลองถูกตั้งไว้ตอนเชื่อมต่อกล้องครั้งแรก

In [ ]:
SOURCE = "emu"          # เปลี่ยนเป็น "basler" เมื่อต่อกล้องจริง

frame = grab_one(SOURCE, image="shapes.png", exposure_us=5000)
draw, result, mask, n_all, n_big = process(frame, 0, 10, 1000)
print("ขนาดภาพจากกล้อง:", frame.shape, "  นับชิ้นแดงใหญ่ได้:", n_big)
show(frame, result, titles=["ภาพจากกล้อง", "ผลการนับ"])

### อ่านข้อความที่ขึ้นมา

ระหว่างรัน โปรแกรมจะพิมพ์ทุกค่าที่ตั้งให้กล้อง เพื่อให้รู้ว่ากล้องอยู่ในสภาพไหนก่อนถ่าย

| ข้อความ | ความหมาย |
|---|---|
| `เชื่อมต่อกล้อง: Emulation` | ต่อกล้องสำเร็จ (กล้องจริงจะขึ้นชื่อรุ่น เช่น `acA2440-20gc`) |
| `ตั้ง TestImageSelector = Off` | ปิดภาพทดสอบลายเส้นของกล้องจำลอง |
| `ตั้ง ImageFilename = .../shapes.png` | บอกกล้องจำลองว่าจะส่งภาพไฟล์ไหน |
| `ตั้ง ImageFileMode = On` | เปิดโหมดส่งภาพจากไฟล์ |
| `ตั้ง PixelFormat = BGR8Packed` | ให้กล้องส่งภาพ**สี** (ค่าเริ่มต้นของกล้องจำลองคือ Mono8 ภาพขาวดำ ซึ่งแยกสีด้วย HSV ไม่ได้) |
| `ตั้ง ExposureAuto = Off` | **ปิดการปรับแสงอัตโนมัติ** ภาพทุกภาพจะได้ความสว่างเท่ากัน |
| `ตั้ง ExposureTime = 5000` | เวลาเปิดรับแสง 5000 ไมโครวินาที |
| `ตั้ง GainAuto = Off` | ปิดการขยายสัญญาณอัตโนมัติ |
| `ตั้ง Gain = 0` | ไม่ขยายสัญญาณ จุดรบกวนน้อยที่สุด |
| `ขนาดภาพจากกล้อง: (480, 640, 3)` | ได้ภาพสูง 480 กว้าง 640 มี 3 ชั้นสี |
| `นับชิ้นแดงใหญ่ได้: 5` | ผลการนับ ตรงกับข้อ 2 ที่ใช้ภาพจากไฟล์ |

ถ้าเห็นคำว่า **(ข้าม ...)** แปลว่ากล้องรุ่นนั้นไม่มีค่านั้น ไม่ใช่ error เช่น กล้อง acA2440-20gc ในห้องใช้ชื่อ `ExposureTimeAbs` และ `GainRaw` แทน โปรแกรมจะลองชื่อให้เอง

**กล้องจำลอง vs กล้องจริง:** กล้องจำลองรับค่า Exposure และ Gain ได้ แต่**ภาพไม่เปลี่ยน** ต้องใช้กล้องจริงจึงเห็นภาพสว่างหรือมืดลง

## กล้องจริง: ต้องจูนใหม่
ภาพจริงสีไม่สดเท่าภาพไฟล์ และภาพใหญ่กว่า (5 MP) ให้ลองลด `s_min`, `v_min` และเพิ่ม `min_area` ใน cell ข้อ 2